In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/bronze/raw_landing"

q = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{base}/_schemas/events")
    .load(f"{base}/events/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .writeStream
    .option("checkpointLocation", f"{base}/_checkpoints/events")
    .trigger(availableNow=True)
    .toTable("workspace.bronze.events"))

q.awaitTermination()

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT event_id) FROM workspace.bronze.events;